# Equal-sample ablation (`tab:eqsample`), Texture Mixer coupling, alignment study, evidence package, KID audit

Renders `tab:eqsample`'s matched-rendering images (hard / soft-OT at two epsilons, one shared 1,024-sample cloud); Texture Mixer's coupling ablation at 120 pairs (`tmc/*`); the cyclic-shift alignment study behind `fig:app-alignshift` and the `\Align*` macros (cell 8); and an audit that corrects the KID reference-term bug for several other notebooks' cached scores (`finishing_audit/*_corrected.csv`).

**Needs:** DTD, the CNT checkpoint, the Texture Mixer checkpoint.

**Maintained runnable path:** `experiments/cnt_matched_200.py` reproduces `tab:eqsample`'s renders; `experiments/alignment.py` reproduces the alignment study. Kept for provenance -- this is also the only record of the KID reference-term audit methodology (see INVENTORY.md section 4b).

# Finishing experiments — the reviewer's list, in one notebook

| # | experiment | phase | writes |
|---|---|---|---|
| 1 | **TM coupling ablation** on the 120 pairs: native lerp, target-mean, entropic OT on an ε grid (ε chosen by retained variance, Sinkhorn iterated to marginal error < 1e-3), hard assignment; encodings, global-latent lerp, source grid and generator identical | TF1 (cells 1–3) | `tm_scale1000/tmc/<rule>/`, `tm_coupling_stats.csv` |
| 2 | **Canonical rescoring** of those outputs: KID vs the fixed reference with paired m-of-n subsampling (identical pair draws across rules), descriptive FID, Inception/DINO realism, signed placement, balance error, paired Wilcoxon | torch (cell 5) | `tm_coupling_scores.csv`, `tm_coupling_summary.json` |
| 3 | **Equal-sample hard vs soft in CNT** on the 200 ablation pairs: the same 1,024 source/target vectors for Hungarian and Sinkhorn (ε = 0.005, 0.05), same nn extension, source conditioning and decoder; midpoint KID/FID/realism/balance and target gap at t = 1 | torch (cell 6) | `matching_ablation/eqsample/`, `eqsample_scores.csv` |
| 4 | **Spatial alignment isolated**: 40 seamless images (mirror-tiled), B = cyclic shift of A by 0 / 4 / 16 / 64 px, same texture distribution throughout; field-linear vs Gaussian vs OT at t = .25 / .5 / .75; deviation from the source's reconstruction, high-frequency retention, realism; visual progression | torch (cell 7) | `alignment/`, `alignment_scores.csv`, figures |
| 5 | **Evidence package**: per-pair scores, seeds, solver diagnostics, pair manifests, summary tables in one folder | torch (cell 8) | `evidence/` |

Run cells 1–3, **Runtime → Restart session**, then cells 4–8. Every loop is resumable through its CSV.

Why ε is chosen by retained variance (item 1): the paper's ε = 0.05 is 1.8× the median cost in CNT space and retains 38% of the target variance; the same rule in TM's latent gives a uniform plan (retained variance 0.2%), i.e. the target-mean coupling. The grid records marginal error, row entropy and retained variance for each ε and the scoring cell reports all of them; the row nearest 38% is marked as the paper-matched setting.


In [ ]:
# ============================================================
# PHASE 1 (TensorFlow), CELL 1 — Texture Mixer setup, verbatim. No torch in this phase.
# ============================================================
import os, sys, types, importlib, importlib.util, importlib.machinery, glob
import numpy as np
from PIL import Image, ImageFile
import matplotlib.pyplot as plt
ImageFile.LOAD_TRUNCATED_IMAGES = True

from google.colab import drive
drive.mount('/content/drive')
BASE = os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
TM_STORE = f'{BASE}/texturemixer'
TM_OUT = f'{TM_STORE}/midpoints'
os.makedirs(TM_OUT, exist_ok=True)

# --- repo ---
os.chdir('/content')
if not os.path.isdir('/content/TextureMixer'):
    !git clone -q https://github.com/ningyu1991/TextureMixer.git
if not os.path.isdir('/content/TextureMixer/tensorflow_vgg'):
    !git clone -q https://github.com/machrisaa/tensorflow-vgg.git /content/TextureMixer/tensorflow_vgg
os.chdir('/content/TextureMixer'); sys.path.insert(0, '/content/TextureMixer')

# --- Python 3.12 removed `imp` ---
if 'imp' not in sys.modules:
    _imp = types.ModuleType('imp')
    def _load_source(name, pathname, file=None):
        spec = importlib.util.spec_from_file_location(name, pathname)
        mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod
        spec.loader.exec_module(mod); return mod
    def _find_module(name, path=None):
        spec = importlib.machinery.PathFinder().find_spec(name, path)
        if spec is None: raise ImportError(name)
        return (None, spec.origin, ('.py','r',1))
    _imp.load_source=_load_source; _imp.find_module=_find_module
    _imp.new_module=lambda n: types.ModuleType(n); _imp.reload=importlib.reload
    _imp.acquire_lock=lambda: None; _imp.release_lock=lambda: None; _imp.PY_SOURCE=1
    sys.modules['imp'] = _imp
    print('imp shim installed')

# --- TF1 graph mode on TF2 ---
import tensorflow as _tf2
print('TF', _tf2.__version__, '| GPU:', _tf2.config.list_physical_devices('GPU'))
tf1 = _tf2.compat.v1
tf1.disable_eager_execution(); tf1.disable_v2_behavior()
tf1.logging.set_verbosity(tf1.logging.ERROR)
if not hasattr(tf1, 'Dimension'): tf1.Dimension = getattr(_tf2, 'Dimension', int)

!pip install -q tf_slim
import tf_slim as _slim
_contrib = types.ModuleType('tensorflow.contrib')
_contrib.slim=_slim; _contrib.framework=_slim; _contrib.layers=_slim
sys.modules['tensorflow.contrib']=_contrib
sys.modules['tensorflow.contrib.slim']=_slim
sys.modules['tensorflow.contrib.framework']=_slim
sys.modules['tensorflow.contrib.layers']=_slim
tf1.contrib=_contrib
sys.modules['tensorflow'] = tf1                 # repo does `import tensorflow as tf`
print('tf_slim wired into tf.contrib')

# --- Python-3 syntax fixes in the repo ---
_p='/content/TextureMixer/tfutil.py'; _s=open(_p).read()
_s=_s.replace('v.shape.ndims is 0','v.shape.ndims == 0').replace('v.shape.ndims is 1','v.shape.ndims == 1')
open(_p,'w').write(_s)

# --- checkpoint (cached on Drive) ---
CKPT = f'{TM_STORE}/earth_texture/network-final.pkl'
if not os.path.exists(CKPT):
    !pip install -q gdown
    !gdown -q 1ObAFBPGaRJFo11LUa0qNhRX14nTEWKC1 -O /content/earth.zip
    !cd /content && unzip -oq earth.zip -d "{TM_STORE}"
print('checkpoint present:', os.path.exists(CKPT))

# --- load networks (9-tuple, bind by name) ---
import tfutil, misc
sess = tfutil.create_session(force_as_default=True)
obj = misc.load_pkl(CKPT)
nets = {getattr(o,'name',f'_{i}'): o for i,o in enumerate(obj)}
print('components:', list(nets))
Es_zg, Es_zl, Gs = nets['Es_zg'], nets['Es_zl'], nets['Gs']
print('LOADED OK — Gs', Gs.input_shapes, '->', Gs.output_shapes)

In [ ]:
# ============================================================
# PHASE 1, CELL 2 — Texture Mixer encode/generate, verbatim
# ============================================================
import random
TM_RES = 128
LATENT_IDX = 0          # 0 = mean (verified); 1 = second moment

def tm_prep(pil):
    a = np.asarray(pil.convert('RGB').resize((TM_RES,)*2), dtype=np.float32)
    return (a/127.5 - 1.0).transpose(2,0,1)[None]

def tm_to_pil(arr):
    return Image.fromarray(np.clip((arr[0].transpose(1,2,0)+1.0)*127.5,0,255).astype(np.uint8))

def tm_encode(pil):
    x = tm_prep(pil)
    return (Es_zg.run(x, return_as_list=True)[LATENT_IDX],
            Es_zl.run(x, return_as_list=True)[LATENT_IDX])

def tm_generate(zg, zl):
    zg_t = np.tile(zg, (1,1,zl.shape[2],zl.shape[3]))
    return tm_to_pil(Gs.run(zg_t, zl, return_as_list=True)[0])

def texturemixer_interp(A, B, weight=0.5):
    zgA, zlA = tm_encode(A); zgB, zlB = tm_encode(B)
    return tm_generate((1-weight)*zgA + weight*zgB, (1-weight)*zlA + weight*zlB)

# --- DTD (a few reads, with retry for Drive EIO) ---
DTD_ROOT = f'{BASE}/dtd/images'
cats = sorted([c for c in os.listdir(DTD_ROOT) if os.path.isdir(f'{DTD_ROOT}/{c}')])
def dtd_path(c,s): return random.Random(s).choice(sorted(glob.glob(f'{DTD_ROOT}/{c}/*.jpg')))
def safe_open(p, tries=4):
    import time as _t
    for i in range(tries):
        try:
            im = Image.open(p); im.load(); return im.convert('RGB')
        except OSError:
            if i == tries-1: raise
            _t.sleep(0.5*(i+1))
print(f'{len(cats)} categories')

# --- reconstruction sanity check ---
probe = safe_open(dtd_path(cats[20], 7))
zg, zl = tm_encode(probe)
fig, ax = plt.subplots(1,2, figsize=(6,3.2))
ax[0].imshow(probe.resize((TM_RES,)*2)); ax[0].set_title('input'); ax[0].axis('off')
ax[1].imshow(tm_generate(zg,zl)); ax[1].set_title('Gs(E(input))'); ax[1].axis('off')
plt.tight_layout(); plt.show()
print('>>> if the right panel resembles the left, continue to cell 3')

In [ ]:
# ============================================================
# PHASE 1, CELL 3 — TM COUPLING ABLATION on the 120 pairs. One encoding per image; zg lerped as TM does; only the zl coupling changes.
#   rules: lerp (positional, = TM native) | mean (U_i = mean(Y)) | ot_r{eps_rel} for eps_rel in EPS_REL (eps = eps_rel * median cost) | hard (Hungarian)
# ============================================================
import numpy as np, pandas as pd, time
from scipy.optimize import linear_sum_assignment
KS120=sorted(int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k); OUT_ROOT=f'{BASE}/tm_scale1000'; TMC=f'{OUT_ROOT}/tmc'
EPS_REL=[0.3,0.1,0.03,0.01,0.003]; RULES=['lerp','mean']+[f'ot_r{e}' for e in EPS_REL]+['hard']
for r_ in RULES: os.makedirs(f'{TMC}/{r_}',exist_ok=True)
def lse(M,ax): mx=M.max(ax,keepdims=True); return (mx+np.log(np.exp(M-mx).sum(ax,keepdims=True))).squeeze(ax)
def sink_log(C,eps,tol=1e-3,max_it=5000):
    """log-domain Sinkhorn, uniform marginals; iterate until relative marginal error < tol. returns plan, err, iters"""
    n,m=C.shape; la,lb=-np.log(n),-np.log(m); f=np.zeros(n); g=np.zeros(m); it=0
    while it<max_it:
        for _ in range(50): f=eps*(la-lse((g[None,:]-C)/eps,1)); g=eps*(lb-lse((f[:,None]-C)/eps,0)); it+=1
        P=np.exp((f[:,None]+g[None,:]-C)/eps); err=max(np.abs(P.sum(1)*n-1).max(),np.abs(P.sum(0)*m-1).max())
        if err<tol: break
    return P,float(err),it
def enc_np(pil): x=tm_prep(pil); return Es_zg.run(x,return_as_list=True)[LATENT_IDX], Es_zl.run(x,return_as_list=True)[LATENT_IDX]
def gen_np(zg,zl): zg_t=np.tile(zg,(1,1,zl.shape[2],zl.shape[3])); return tm_to_pil(Gs.run(zg_t,zl,return_as_list=True)[0])
if 'tm_to_pil' not in globals():
    def tm_to_pil(arr): a=np.asarray(arr); a=a[0] if a.ndim==4 else a; return Image.fromarray(np.clip((a.transpose(1,2,0)+1)*127.5,0,255).astype(np.uint8))
CSV=f'{BASE}/tm_coupling_stats.csv'; rows=pd.read_csv(CSV).to_dict('records') if os.path.exists(CSV) else []; have={(r['k'],r['rule']) for r in rows}; t0=time.time()
for n_,k in enumerate(KS120):
    if all((k,r_) in have for r_ in RULES) and all(os.path.exists(f'{TMC}/{r_}/{k:04d}.png') for r_ in RULES): continue
    A=Image.open(f'{OUT_ROOT}/A/{k:04d}.png').convert('RGB'); B=Image.open(f'{OUT_ROOT}/B/{k:04d}.png').convert('RGB')
    (gA,lA),(gB,lB)=enc_np(A),enc_np(B); C_,H,W=lA.shape[1:]; X=lA[0].reshape(C_,-1).T.astype(np.float64); Y=lB[0].reshape(C_,-1).T.astype(np.float64)
    Cm=((X[:,None,:]-Y[None,:,:])**2).sum(-1); medC=float(np.median(Cm)); zg_mid=0.5*(gA+gB); varY=Y.var(0).sum()
    T={'lerp':(Y,dict()),'mean':(np.tile(Y.mean(0),(len(X),1)),dict())}
    for e in EPS_REL:
        P,err,it=sink_log(Cm,e*medC); U=(P/P.sum(1,keepdims=True))@Y; q=P/P.sum(1,keepdims=True); Hrow=float(-(q*np.log(q+1e-300)).sum(1).mean())/np.log(len(Y))
        T[f'ot_r{e}']=(U,dict(eps=e*medC,eps_rel=e,marg_err=err,iters=it,H_row_norm=Hrow,ot_cost=float((P*Cm).sum())))
    r,c=linear_sum_assignment(Cm); Uh=np.empty_like(X); Uh[r]=Y[c]; T['hard']=(Uh,dict(hard_cost=float(Cm[r,c].mean())))
    for name,(U,info) in T.items():
        if (k,name) not in have: rows.append(dict(k=k,rule=name,med_cost=medC,var_ret=float(U.var(0).sum()/varY),**info))
        fp=f'{TMC}/{name}/{k:04d}.png'
        if not os.path.exists(fp): gen_np(zg_mid,(0.5*X+0.5*U).T.reshape(1,C_,H,W).astype(np.float32)).save(fp)
    if n_%10==0: pd.DataFrame(rows).to_csv(CSV,index=False); print(f'{n_+1}/120 {time.time()-t0:.0f}s')
pd.DataFrame(rows).to_csv(CSV,index=False); S=pd.DataFrame(rows)
print('\nretained target variance / normalised row entropy / max marginal error, medians per rule:')
print(S.groupby('rule')[['var_ret','H_row_norm','marg_err','iters']].median().round(4).to_string())
tgt=0.38; vr=S[S.rule.str.startswith('ot_')].groupby('rule').var_ret.median(); print(f'\npaper-matched eps (retained variance nearest {tgt}):',vr.sub(tgt).abs().idxmin(),'| any rule with marg_err>1e-3:',S[S.marg_err>1e-3].rule.unique().tolist() if 'marg_err' in S else [])
print('>>> PHASE 1 done. Runtime -> Restart session, then run cells 4-8.')


## Restart the runtime here (Runtime → Restart session), then run cells 4–8.


In [ ]:
# ============================================================
# PHASE 2 (torch), CELL 4 — canonical SETUP (fast: DTD from Drive zip, detectron2 only if the import fails) — canonical protocol (finish_day / mechanism cell 0, verbatim) + pinned CNT commit
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile,hashlib,itertools
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
assert torch.cuda.is_available(),'switch runtime to GPU'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'
MECH=f'{BASE}/mechanism'; os.makedirs(MECH,exist_ok=True); FIGS=f'{MECH}/figs'; os.makedirs(FIGS,exist_ok=True)
TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
t0=time.time()

# --- DTD ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system('rm -rf /content/_dtd'); os.system(f'unzip -qo "{ZIP}" -d /content/_dtd')
    inner=os.listdir('/content/_dtd'); src='/content/_dtd/images' if 'images' in inner else f'/content/_dtd/{inner[0]}'
    os.system(f'rm -rf {LOCAL} && mv "{src}" {LOCAL}')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24 && cd cnt-siga24 && git checkout -q 07269302610c92ce38e2dbbe31c492fd09e1c73e')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
CNT_COMMIT=os.popen(f'cd {CNT} && git rev-parse HEAD').read().strip()
print(f'model ready {time.time()-t0:.0f}s | FMS={FMS} | CNT commit {CNT_COMMIT[:10]}')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def _pyr(si):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(FMS): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return sp
def build_style_maps(bf,si,fms):
    sp=_pyr(si); return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def decode_stagewise(blob_ref,sm_by_stage):
    """same decoder, but stage i receives sm_by_stage[i] (dict size->map). For E5 per-stage swaps."""
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm_by_stage[i][fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def sm_from_field(Z,L,H,W,Cc):
    it=Z.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
def _dec(d,Z,L,H,W,Cc,layout='a'): return decode_with_style_maps(d["blob_"+layout],sm_from_field(Z.float(),L,H,W,Cc))
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def pair_imgs(k): return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

# --- canonical Sinkhorn (as used in the paper) + converged log-domain Sinkhorn with duals ---
def sinkhorn_plan(x,y,eps,iters=ITERS):
    Cm=torch.cdist(x,y).pow(2); K=torch.exp(-Cm/eps)
    u=torch.ones(len(x),device=x.device)/len(x); v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u); b=torch.ones_like(v)
    for _ in range(iters): a=u/(K@b+1e-30); b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:], Cm
def sinkhorn_log(x,y,eps,iters=3000,tol=1e-6):
    """balanced, uniform marginals, float64 log-domain; returns P, duals f,g, cost, marginal residual, iterations."""
    X=x.double(); Y=y.double(); Cm=torch.cdist(X,Y).pow(2); n,m=Cm.shape
    la=-math.log(n); lb=-math.log(m); f=torch.zeros(n,device=X.device,dtype=torch.float64); g=torch.zeros(m,device=X.device,dtype=torch.float64)
    for it in range(1,iters+1):
        f=eps*(la-torch.logsumexp((g[None,:]-Cm)/eps,1)); g=eps*(lb-torch.logsumexp((f[:,None]-Cm)/eps,0))
        if it%10==0:
            logP=(f[:,None]+g[None,:]-Cm)/eps; res=float((torch.exp(logP).sum(1)-1/n).abs().max()*n)
            if res<tol: break
    logP=(f[:,None]+g[None,:]-Cm)/eps; P=torch.exp(logP)
    res=float(max((P.sum(1)-1/n).abs().max()*n,(P.sum(0)-1/m).abs().max()*m))
    return P,f,g,Cm,res,it
def plan_stats(P):
    P=P.double(); n=P.shape[0]; q=P/(P.sum(1,keepdim=True)+1e-300)
    Hrow=float(-(q*torch.log(q+1e-300)).sum(1).mean()); Htot=float(-(P*torch.log(P+1e-300)).sum())
    return dict(H_row=Hrow,H_total=Htot,MI_nats=math.log(n)-Hrow)
def natural_extension(x_all,y,g,eps,chunk=1024):
    """T(x)=sum_j q_j(x) y_j, q_j ∝ exp((g_j-||x-y_j||²)/eps) — dual-potential extension of a converged plan."""
    out=[]
    for i in range(0,len(x_all),chunk):
        Cm=torch.cdist(x_all[i:i+chunk].double(),y.double()).pow(2); q=torch.softmax((g[None,:]-Cm)/eps,1); out.append(q@y.double())
    return torch.cat(out)

# --- matching rules -> sampled targets (M,d); full field via canonical nn extension ---
def _targets(sf,tf,rule,seed,eps=EPS,M=NSUB,n_exact=1024,plan_kind='canonical'):
    seed=int(seed)   # numpy int64 from pandas breaks torch manual_seed
    N=sf.shape[0]; i_s=_sub(N,seed,'s')[:M]; i_t=_sub(N,seed,'t')[:M]; src,tgt=sf[i_s],tf[i_t]; info={}
    if rule in('ot','ot_sharp','ot_soft'):
        e={'ot':eps,'ot_sharp':0.005,'ot_soft':0.5}[rule] if rule!='ot' else eps
        if plan_kind=='canonical': P,_=sinkhorn_plan(src,tgt,e); info=dict(eps=e,**plan_stats(P))
        else: P,f,g,_,res,it=sinkhorn_log(src,tgt,e); info=dict(eps=e,res=res,iters=it,**plan_stats(P)); info['g']=g
        tt=((P/(P.sum(1,keepdim=True)+1e-30))@tgt.double()).float() if P.dtype==torch.float64 else (P/(P.sum(1,keepdim=True)+1e-30))@tgt
    elif rule=='exact':
        g_=torch.Generator(device=device).manual_seed(seed+11)
        js=torch.randperm(M,generator=g_,device=device)[:n_exact]; is_=torch.randperm(M,generator=g_,device=device)[:n_exact]
        r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
        tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
        ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]; info=dict(n_exact=n_exact)
    elif rule=='nn': tt=tgt[torch.cdist(src,tgt).argmin(1)]
    elif rule=='gauss':
        X,Y=src.double(),tgt.double(); mx,my=X.mean(0),Y.mean(0); Xc=X-mx; Yc=Y-my; Cc=X.shape[1]; I=torch.eye(Cc,device=device,dtype=torch.float64); ridge=1e-4
        Sa=(Xc.T@Xc)/(len(X)-1)+ridge*I; Sb=(Yc.T@Yc)/(len(Y)-1)+ridge*I
        def ps(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.sqrt())@V.T
        def pis(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.rsqrt())@V.T
        Sh=ps(Sa,ridge); Sih=pis(Sa,ridge); Amap=Sih@ps(Sh@Sb@Sh,0.0)@Sih; Amap=.5*(Amap+Amap.T)
        tt=(my+Xc@Amap.T).float(); info=dict(A=Amap,mu=(mx,my))
    elif rule=='uniform': tt=tgt.mean(0,keepdim=True).expand_as(src).clone()
    elif rule=='random':
        g_=torch.Generator(device=device).manual_seed(seed+7); tt=tgt[torch.randperm(M,generator=g_,device=device)]
    elif rule=='meanshift': tt=src+(tgt.mean(0)-src.mean(0))      # CNT-style native mean shift, adapted to the interpolation contract
    else: raise ValueError(rule)
    return i_s,tt,info
def field(d,rule,seed,**kw):
    seed=int(seed)
    """full-grid target field U (N,d) with the canonical nn extension; also returns X,Y grids and plan info."""
    L,Cc,H,W,sf,tf=_grids(d); i_s,tt,info=_targets(sf,tf,rule,seed,**kw)
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return dict(U=tt[nn],X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,i_s=i_s,tt=tt,nn=nn,info=info)
def zt(fd,t): return (1-t)*fd['X']+t*fd['U']
def decode_field(d,fd,Z): return _dec(d,Z,fd['L'],fd['H'],fd['W'],fd['Cc'])

# --- fixed-layout subspace of A: X = W_A F_A ---
def splat_W(d,side='a'):
    sp=_pyr(d['si_'+side]); Wm=sp[max(FMS)][0]; K=Wm.shape[0]
    return Wm.reshape(K,-1).T.contiguous().double(), d['bf_'+side][0].double()      # (N,K),(K,d)
def projector(Wm,tol=1e-6):
    Uw,S,_=torch.linalg.svd(Wm,full_matrices=False); keep=S>S[0]*tol; return Uw[:,keep], int(keep.sum())
def proj(Q,U): U=U.double(); return Q@(Q.T@U)
def e_sub(Q,U):
    U=U.double(); Uc=U-U.mean(0,keepdim=True); off=U-proj(Q,U)
    return dict(e_sub=float(off.norm()/(Uc.norm()+1e-12)),off_rms=float(off.norm()/math.sqrt(U.numel())),cen_rms=float(Uc.norm()/math.sqrt(U.numel())))
def e_aff(X,U,seed=0):
    """affine fit U≈XA+b on half the rows, residual on held-out rows."""
    X=X.double(); U=U.double(); N=len(X); g=torch.Generator(device=X.device).manual_seed(seed); p=torch.randperm(N,generator=g,device=X.device); tr,te=p[:N//2],p[N//2:]
    Xa=torch.cat([X,torch.ones(N,1,device=X.device,dtype=torch.float64)],1)
    G=Xa[tr].T@Xa[tr]; sol=torch.linalg.solve(G+1e-6*torch.trace(G)/G.shape[0]*torch.eye(G.shape[0],device=G.device,dtype=G.dtype),Xa[tr].T@U[tr]); R=U[te]-Xa[te]@sol; Uc=U[te]-U[te].mean(0,keepdim=True)
    return dict(e_aff=float(R.norm()/(Uc.norm()+1e-12)),aff_rms=float(R.norm()/math.sqrt(R.numel())))
def commutation_err(Q,Wm,Fm,Tfun):
    """J = T(W F) - W T(F), for a frozen map T applied to blob vectors (out-of-support warning if far from cloud)."""
    X=(Wm@Fm); TX=Tfun(X.float()).double(); TF=Tfun(Fm.float()).double(); J=TX-Wm@TF
    return dict(comm=float(J.norm()/(TX-TX.mean(0)).norm()),comm_rms=float(J.norm()/math.sqrt(J.numel())))
def neighbor_disc(U,H,W):
    G=U.reshape(H,W,-1); return float(((G[:,1:]-G[:,:-1]).norm(dim=-1).mean()+(G[1:]-G[:-1]).norm(dim=-1).mean())/2)
def var_ret(U,Y): return float(U.double().var(0).sum()/Y.double().var(0).sum())
def interaction_scale(X,Y):
    """RMS of -2 Xc Yc^T (centered interaction) vs raw median cost: the calibration the plan asks for."""
    Xc=X.double()-X.double().mean(0); Yc=Y.double()-Y.double().mean(0); M_=-2*Xc@Yc.T
    return dict(inter_rms=float(M_.pow(2).mean().sqrt()),cost_med=float(torch.cdist(X.double(),Y.double()).pow(2).median()))

# --- observers (canonical) ---
import timm, lpips as lpips_lib
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
lp=lpips_lib.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p:(torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
SCORE_RES=128
def at128(pil): return pil.convert('RGB').resize((SCORE_RES,SCORE_RES))
ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def kid_point(X,Y,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(len(X),len(Y)); x=X[:m]; y=Y[rng.choice(len(Y),m,replace=False)]
    kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
KS120=[int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k]; KSET=set(KS120)
DEV=[int(p_['k']) for p_ in MAN_ALL if int(p_['k']) not in KSET][:24]          # development set: pair-disjoint from the 120 confirmation pairs
print('dev pairs:',DEV); print('confirmation pairs:',len(KS120))

# --- per-pair scoring bundle for a decoded midpoint (human-study proxies) ---
_anc={}
def anchors(k):
    if k in _anc: return _anc[k]
    A,B=pair_imgs(k); rA=to_pil(make_recon(A)); rB=to_pil(make_recon(B)); f=dino_feats([at128(rA),at128(rB)])
    _anc[k]=dict(fA=f[0],fB=f[1],dAB=float(np.linalg.norm(f[0]-f[1])),rA=rA,rB=rB); return _anc[k]
def reencode_field(pil):
    e=cnt_load(pil); ba,bfa,sia=get_gen_inputs(e); sm=build_style_maps(bfa,sia,FMS); L=max(FMS); A=sm[L][0]; Cc=A.shape[0]
    return A.permute(1,2,0).reshape(-1,Cc)
def score_image(k,pil,Z=None,t=None):
    a=anchors(k); im=at128(pil); fi=inc_feats([im])[0]; fd=dino_feats([im])[0]
    r=dict(real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]),
           dA=float(np.linalg.norm(fd-a['fA'])/a['dAB']),dB=float(np.linalg.norm(fd-a['fB'])/a['dAB']))
    r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    if Z is not None:
        Xr=reencode_field(pil); r['roundtrip']=float((Xr.double()-Z.double()).norm()/(Z.double().norm()+1e-12))
    return r,fi,fd
_cache_d={}
def get_d(k):
    if k not in _cache_d:
        if len(_cache_d)>64: _cache_d.clear()
        A,B=pair_imgs(k); _cache_d[k]=build_d(cnt_load(A),cnt_load(B))
    return _cache_d[k]
def save_rows(rows,path): pd.DataFrame(rows).to_csv(path,index=False)
def load_rows(path,keys):
    if os.path.exists(path):
        df=pd.read_csv(path); return df.to_dict('records'), set(tuple(r[c] for c in keys) for r in df.to_dict('records'))
    return [],set()
print(f'setup done in {time.time()-t0:.0f}s')
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False; torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
print('precision: fp32, deterministic cuDNN')


In [ ]:
# ============================================================
# 5. CANONICAL RESCORING of the TM coupling outputs — same reference, models and realism implementation as the paper; paired uncertainty
# ============================================================
TMC=f'{SCALE}/tmc'; S1=pd.read_csv(f'{BASE}/tm_coupling_stats.csv'); RULES=['lerp','mean']+sorted([r for r in S1.rule.unique() if r.startswith('ot_')],key=lambda s:-float(s[4:]))+['hard']
if os.path.isdir(f'{SCALE}/mid'): RULES=['mid_pipeline']+RULES
def rule_dir(r): return f'{SCALE}/mid' if r=='mid_pipeline' else f'{TMC}/{r}'
REF=ref_inc.astype(np.float64); _k=((REF@REF.T/REF.shape[1]+1)**3); KYY=(_k.sum()-len(REF))/(len(REF)*(len(REF)-1)); del _k
def kid_ub(X): dd=X.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3; m=len(X); kxx=kk(X,X); return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+KYY-2*kk(X,REF).mean())
from scipy import linalg as _sl
def fid(X): mx,my=X.mean(0),REF.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(REF,rowvar=False); cm=_sl.sqrtm(Sx@Sy); cm=cm.real if np.iscomplexobj(cm) else cm; return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*cm))
CSV5=f'{BASE}/tm_coupling_scores.csv'; rows,have=load_rows(CSV5,['k','rule']); FE={}; t0=time.time()
for n_,k in enumerate(KS120):
    for r_ in RULES:
        fp=f'{rule_dir(r_)}/{k:04d}.png'
        if not os.path.exists(fp): continue
        s,fi,fd=score_image(k,Image.open(fp).convert('RGB')); FE[(r_,k)]=fi
        if (k,r_) not in have: rows.append(dict(k=k,rule=r_,**s))
    if n_%20==0: save_rows(rows,CSV5); print(f'{n_+1}/120 {time.time()-t0:.0f}s')
save_rows(rows,CSV5); J=pd.DataFrame(rows); ks=sorted(set(J.k)); J['bal']=(J.place-0.5).abs()
X={r_:np.stack([FE[(r_,k)] for k in ks]).astype(np.float64) for r_ in RULES if all((r_,k) in FE for k in ks)}
B_,m=2000,100; rng=np.random.RandomState(20260911); draws={r_:[] for r_ in X}
for _ in range(B_):
    idx=rng.choice(len(ks),m,replace=False)                     # identical pair draw for every rule; reference fixed
    for r_ in X: draws[r_].append(kid_ub(X[r_][idx]))
T=J.groupby('rule')[['real_inc','real_dino','place','bal']].median().loc[[r for r in RULES if r in X]]
T['KID']=[kid_ub(X[r_]) for r_ in T.index]; T['FID']=[fid(X[r_]) for r_ in T.index]
stats=S1.groupby('rule')[['var_ret','H_row_norm','marg_err']].median(); T=T.join(stats,how='left')
def dk(a,b): dv=np.array(draws[a])-np.array(draws[b]); return T.loc[a,'KID']-T.loc[b,'KID'],float(np.percentile(dv,2.5)),float(np.percentile(dv,97.5))
pv=J.pivot(index='k',columns='rule',values='real_dino'); pb=J.pivot(index='k',columns='rule',values='bal')
for r_ in T.index:
    if r_=='lerp': continue
    d_,lo,hi=dk(r_,'lerp'); T.loc[r_,'dKID_vs_lerp']=d_; T.loc[r_,'dKID_CI']=f'[{lo:+.4f},{hi:+.4f}]'
    x=(pv[r_]-pv['lerp']).dropna(); T.loc[r_,'dReal_vs_lerp']=x.median(); T.loc[r_,'p_real']=wilcoxon(x).pvalue if np.abs(x).sum()>0 else 1.0
    if 'mean' in T.index and r_!='mean': d2,lo2,hi2=dk(r_,'mean'); T.loc[r_,'dKID_vs_mean']=d2; T.loc[r_,'dKID_mean_CI']=f'[{lo2:+.4f},{hi2:+.4f}]'
print('=== TM coupling ablation, canonical 128 px protocol, n=%d pairs ==='%len(ks)); print(T.round(4).to_string())
star=(T[T.index.str.startswith('ot_')].var_ret-0.38).abs().idxmin(); print(f'\npaper-matched entropic row: {star} (retained variance {T.loc[star,"var_ret"]:.3f})')
print('read: does transport beat averaging?  ->  dKID_vs_mean for',star,'and hard; CI excluding 0 = yes.  |  does the improvement survive eps? -> the ot_* rows.')
json.dump(dict(table=T.reset_index().to_dict('records'),paper_matched=star,m_of_n=dict(B=B_,m=m,n=len(ks)),seed=20260911),open(f'{BASE}/tm_coupling_summary.json','w'),indent=1,default=str)


In [ ]:
# ============================================================
# 6. EQUAL-SAMPLE HARD vs SOFT in CNT — 200 ablation pairs, identical 1,024 sampled source/target vectors for Hungarian and Sinkhorn
#    (eps = 0.005, 0.05, converged), same nn extension, source conditioning and decoder; midpoint (t=.5) and transported endpoint (t=1)
# ============================================================
KS200=[int(p_['k']) for p_ in MAN_ALL[:200]]; NEQ=1024; EQD=f'{OUT}/eqsample'; RULES6=['hard','ot_e0.005','ot_e0.05']; TS6=[0.5,1.0]
def sink_log_t(C,eps,tol=1e-3,max_it=5000):
    n,m=C.shape; la,lb=-math.log(n),-math.log(m); f=torch.zeros(n,device=device,dtype=C.dtype); g=torch.zeros(m,device=device,dtype=C.dtype); it=0
    while it<max_it:
        for _ in range(50): f=eps*(la-torch.logsumexp((g[None,:]-C)/eps,1)); g=eps*(lb-torch.logsumexp((f[:,None]-C)/eps,0)); it+=1
        P=torch.exp((f[:,None]+g[None,:]-C)/eps); err=float(max((P.sum(1)*n-1).abs().max(),(P.sum(0)*m-1).abs().max()))
        if err<tol: break
    return P,err,it
CSV6=f'{BASE}/eqsample_scores.csv'; rows,have=load_rows(CSV6,['k','rule','t']); FE6={}; t0=time.time()
for r_ in RULES6:
    for t in TS6: os.makedirs(f'{EQD}/{r_}_t{t}',exist_ok=True)
for n_,k in enumerate(KS200):
    if all((k,r_,t) in have for r_ in RULES6 for t in TS6): continue
    d=get_d(k); L,Cc,H,W,sf,tf=_grids(d); N=sf.shape[0]; i_s=_sub(N,k,'s')[:NEQ]; i_t=_sub(N,k,'t')[:NEQ]; src,tgt=sf[i_s],tf[i_t]   # the SAME 1024 for every rule
    Cm=torch.cdist(src.double(),tgt.double()).pow(2); nn=torch.cdist(sf,src).argmin(1); anc=anchors(k); TT={}
    r,c=linear_sum_assignment(Cm.cpu().numpy()); th=torch.empty_like(src); th[torch.as_tensor(r,device=device)]=tgt[torch.as_tensor(c,device=device)]; TT['hard']=(th,dict(cost=float(Cm.cpu().numpy()[r,c].mean())))
    for e in [0.005,0.05]:
        P,err,it=sink_log_t(Cm,e); TT[f'ot_e{e}']=(((P/P.sum(1,keepdim=True))@tgt.double()).float(),dict(marg_err=err,iters=it,H_row=plan_stats(P)['H_row'],var_ret=var_ret(((P/P.sum(1,keepdim=True))@tgt.double()).float(),tgt)))
    for r_,(tt,info) in TT.items():
        U=tt[nn]
        for t in TS6:
            if (k,r_,t) in have: continue
            fp=f'{EQD}/{r_}_t{t}/{k:04d}.png'
            if not os.path.exists(fp): to_pil(_dec(d,(1-t)*sf+t*U,L,H,W,Cc)).save(fp)
            pil=Image.open(fp).convert('RGB'); s,fi,fd=score_image(k,pil); FE6[(r_,t,k)]=fi
            rows.append(dict(k=k,rule=r_,t=t,**s,target_gap=float(np.linalg.norm(fd-anc['fB'])/anc['dAB']),source_gap=float(np.linalg.norm(fd-anc['fA'])/anc['dAB']),**info))
    if n_%20==0: save_rows(rows,CSV6); print(f'{n_+1}/200 {time.time()-t0:.0f}s')
save_rows(rows,CSV6); E=pd.DataFrame(rows); E['bal']=(E.place-0.5).abs()
if os.path.isdir(f'{SCALE}/ot'):                                                # the paper's 4096-sample operator on the same pairs, for reference (midpoint only)
    for k in KS200:
        fp=f'{SCALE}/ot/{k:04d}.png'
        if os.path.exists(fp): s,fi,fd=score_image(k,Image.open(fp).convert('RGB')); FE6[('ot_4096',0.5,k)]=fi
ksm=sorted(set(E.k)); X6={r_:np.stack([FE6[(r_,0.5,k)] for k in ksm]).astype(np.float64) for r_ in RULES6+['ot_4096'] if all((r_,0.5,k) in FE6 for k in ksm)}
rng=np.random.RandomState(1); dr6={r_:[] for r_ in X6}
for _ in range(2000):
    idx=rng.choice(len(ksm),150,replace=False)
    for r_ in X6: dr6[r_].append(kid_ub(X6[r_][idx]))
mid=E[E.t==0.5].groupby('rule')[['real_inc','real_dino','place','bal','var_ret','H_row','marg_err']].median(); mid['KID']=[kid_ub(X6[r_]) if r_ in X6 else np.nan for r_ in mid.index]; mid['FID']=[fid(X6[r_]) if r_ in X6 else np.nan for r_ in mid.index]
end=E[E.t==1.0].groupby('rule')[['target_gap','source_gap','real_dino']].median().add_prefix('t1_'); mid=mid.join(end)
if 'ot_4096' in X6: mid.loc['ot_4096 (paper)','KID']=kid_ub(X6['ot_4096']); mid.loc['ot_4096 (paper)','FID']=fid(X6['ot_4096'])
print('=== equal-sample coupling ablation, n=%d pairs, canonical scoring ==='%len(ksm)); print(mid.round(4).to_string())
for a,b in [('hard','ot_e0.05'),('hard','ot_e0.005'),('ot_e0.005','ot_e0.05')]:
    if a in X6 and b in X6: dv=np.array(dr6[a])-np.array(dr6[b]); print(f'dKID {a} - {b} = {kid_ub(X6[a])-kid_ub(X6[b]):+.4f} [{np.percentile(dv,2.5):+.4f},{np.percentile(dv,97.5):+.4f}]')
pb=E[E.t==0.5].pivot(index='k',columns='rule',values='bal'); pg=E[E.t==1.0].pivot(index='k',columns='rule',values='target_gap')
print('paired hard - ot_e0.05: balance error %+.4f (p=%.2g) | target gap at t=1 %+.4f (p=%.2g)'%((pb.hard-pb['ot_e0.05']).median(),wilcoxon(pb.hard-pb['ot_e0.05']).pvalue,(pg.hard-pg['ot_e0.05']).median(),wilcoxon(pg.hard-pg['ot_e0.05']).pvalue))
mid.to_csv(f'{BASE}/eqsample_summary.csv')


In [ ]:
# ============================================================
# 7. SPATIAL ALIGNMENT ISOLATED — 40 seamless images (mirror-tiled 128 crops -> periodic 256), B = cyclic horizontal shift of A by 0/4/16/64 px.
#    The texture distribution is identical for every shift; only positional correspondence changes. field-linear vs Gaussian vs OT, t = .25/.5/.75.
#    Metrics vs the source's own reconstruction R_A: DINO distance (texture-statistic deviation), LPIPS (positional), HF energy retention, DINO realism.
# ============================================================
ALD=f'{BASE}/alignment'; os.makedirs(ALD,exist_ok=True); SHIFTS=[0,4,16,64]; TS7=[0.25,0.5,0.75]; RULES7=['linear','gauss','ot']; N_IMG=40
files=sorted(glob.glob(f'{LOCAL}/*/*.jpg')); files=[files[i] for i in np.random.RandomState(7).permutation(len(files))]
def seamless(path):
    im=Image.open(path).convert('RGB'); w,h=im.size; r=256/min(w,h); im=im.resize((max(256,round(w*r)),max(256,round(h*r))),Image.BICUBIC); w,h=im.size; c=im.crop(((w-128)//2,(h-128)//2,(w-128)//2+128,(h-128)//2+128))
    out=Image.new('RGB',(256,256)); out.paste(c,(0,0)); out.paste(c.transpose(Image.FLIP_LEFT_RIGHT),(128,0)); out.paste(c.transpose(Image.FLIP_TOP_BOTTOM),(0,128)); out.paste(c.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM),(128,128)); return out
def shift(im,s): a=np.asarray(im); return Image.fromarray(np.roll(a,s,axis=1))
def hf_energy(pil,r0=32):
    g=np.asarray(pil.convert('L'),np.float32)/255.; Fm=np.abs(np.fft.fftshift(np.fft.fft2(g-g.mean())))**2; h,w=g.shape; yy,xx=np.mgrid[:h,:w]; rr=np.hypot(yy-h/2,xx-w/2); return float(Fm[rr>r0].sum()/(Fm.sum()+1e-9))
sel=[f for f in files if min(Image.open(f).size)>=256][:N_IMG]
CSV7=f'{BASE}/alignment_scores.csv'; rows,have=load_rows(CSV7,['img','shift','rule','t']); t0=time.time()
for ii,f in enumerate(sel):
    if all((ii,s,r_,t) in have for s in SHIFTS for r_ in RULES7 for t in TS7): continue
    A=seamless(f); RA=to_pil(make_recon(A)); fRA=dino_feats([at128(RA)])[0]; hfA=hf_energy(RA); ia=cnt_load(A)
    for s in SHIFTS:
        Bs=shift(A,s); d=build_d(ia,cnt_load(Bs)); fds={'gauss':field(d,'gauss',ii),'ot':field(d,'ot',ii)}
        for r_ in RULES7:
            for t in TS7:
                if (ii,s,r_,t) in have: continue
                out=interp_lin_pixel(d,t) if r_=='linear' else decode_field(d,fds[r_],zt(fds[r_],t)); pil=to_pil(out)
                if ii<3: os.makedirs(f'{ALD}/imgs',exist_ok=True); pil.save(f'{ALD}/imgs/img{ii}_s{s}_{r_}_t{t}.png'); (RA.save(f'{ALD}/imgs/img{ii}_recA.png') if not os.path.exists(f'{ALD}/imgs/img{ii}_recA.png') else None); Bs.save(f'{ALD}/imgs/img{ii}_s{s}_B.png')
                fd=dino_feats([at128(pil)])[0]
                rows.append(dict(img=ii,shift=s,rule=r_,t=t,dino_dev=float(np.linalg.norm(fd-fRA)),lpips_recA=lpips_d(pil,RA),hf_ret=hf_energy(pil)/(hfA+1e-9),real_dino=float(realism(fd[None],MANI_DINO)[0])))
    save_rows(rows,CSV7); print(f'{ii+1}/{len(sel)} {time.time()-t0:.0f}s')
AL=pd.DataFrame(rows)
print('=== medians at t=0.5 by shift (rows) and rule: DINO deviation from R_A (lower better), HF retention (1 = source), realism ===')
for m_ in ['dino_dev','hf_ret','real_dino']: print(m_); print(AL[AL.t==0.5].pivot_table(index='shift',columns='rule',values=m_,aggfunc='median').round(3).to_string())
fig,ax=plt.subplots(1,3,figsize=(14,3.8))
for a_,m_ in zip(ax,['dino_dev','hf_ret','real_dino']):
    for r_ in RULES7: g=AL[(AL.t==0.5)&(AL.rule==r_)].groupby('shift')[m_].median(); a_.plot(g.index,g.values,marker='o',label=r_)
    a_.set_xscale('symlog',linthresh=4); a_.set_xlabel('cyclic shift of B (px)'); a_.set_ylabel(m_); a_.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{ALD}/alignment_curves.png',dpi=150); plt.show()
fig,ax=plt.subplots(len(SHIFTS),5,figsize=(10,2.1*len(SHIFTS)))
for i,s in enumerate(SHIFTS):
    ims=[Image.open(f'{ALD}/imgs/img0_recA.png')]+[Image.open(f'{ALD}/imgs/img0_s{s}_{r_}_t0.5.png') for r_ in RULES7]+[Image.open(f'{ALD}/imgs/img0_s{s}_B.png')]
    for j,(t_,im) in enumerate(zip(['recon A','field-linear','Gaussian','OT','B (shifted)'],ims)): ax[i,j].imshow(im); ax[i,j].axis('off'); ax[i,j].set_title(t_ if i==0 else '',fontsize=8)
    ax[i,0].text(-0.08,0.5,f'shift {s}px',transform=ax[i,0].transAxes,rotation=90,va='center',fontsize=8)
plt.tight_layout(); plt.savefig(f'{ALD}/alignment_progression.png',dpi=150,bbox_inches='tight'); plt.show()


In [ ]:
# ============================================================
# 8. EVIDENCE PACKAGE — per-pair scores, seeds, solver settings, pair manifests, summary tables in one folder
# ============================================================
import shutil
EV=f'{BASE}/evidence'; os.makedirs(EV,exist_ok=True)
for f in ['tm_coupling_stats.csv','tm_coupling_scores.csv','tm_coupling_summary.json','eqsample_scores.csv','eqsample_summary.csv','alignment_scores.csv','timing_120.csv','timing_120_tm.csv','texture_distance_probe.csv','texture_distance_scores_v2.csv','pair_distance_ours_vs_tm_120.csv','dists_fidelity_120.csv']:
    p=f'{BASE}/{f}'; (shutil.copy(p,f'{EV}/{f}') if os.path.exists(p) else print('missing:',f))
pd.DataFrame([dict(k=k,fA=os.path.basename(str(p_.get('fA',''))),fB=os.path.basename(str(p_.get('fB','')))) for p_ in MAN_ALL for k in [int(p_['k'])]]).to_csv(f'{EV}/manifest_1000_pairs.csv',index=False)
pd.DataFrame(dict(k=KS120)).to_csv(f'{EV}/pairs_120.csv',index=False); pd.DataFrame(dict(k=[int(p_['k']) for p_ in MAN_ALL[:200]])).to_csv(f'{EV}/pairs_200_ablation.csv',index=False)
for pat in ['ref_*.npy','ref*.json','ref*.csv','*manifest*']:
    for p in glob.glob(f'{OUT}/{pat}'): shutil.copy(p,f'{EV}/{os.path.basename(p)}')
json.dump(dict(cnt_commit=globals().get('CNT_COMMIT','see setup'),img_size=IMG_SIZE,score_res=SCORE_RES,endpoint_files='128x128 PNGs (Texture Mixer native), bicubically upsampled to 256 before CNT encoding',
               sinkhorn=dict(eps=EPS,iters_canonical=ITERS,subsample=NSUB,note='ablations in this notebook iterate log-domain Sinkhorn to marginal error <1e-3'),hard=dict(n_exact=1024),
               seeds=dict(subsample='_sub(N,seed=pair_id,tag)',kid_subsampling=dict(m=100,B=2000,seed=20260911),alignment_images=7),
               reference=dict(n=int(len(ref_inc)),note='ref_inc.npy / ref_dino.npy in OUT; image list copied if present'),
               missing=['GPT-Image-1.5 verbatim prompts and call ledger (not in this package)']),open(f'{EV}/README.json','w'),indent=1)
print('evidence written to',EV); print(sorted(os.listdir(EV)))


In [ ]:
# ============================================================
# AUDIT + REPAIR — run after the existing Phase 2 cells.
# Uses saved images; does not rerun interpolation or overwrite images.
# ============================================================
import os
import json
import time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from scipy.stats import norm

_required = [
    "BASE", "OUT", "SCALE", "FMS", "MAN_ALL", "KS120",
    "ref_inc", "inc_feats", "at128", "get_d", "_grids",
    "_dec", "interp_lin_pixel", "decode_with_style_maps",
    "build_style_maps", "to_pil", "lpips_d",
]
_missing = [name for name in _required if name not in globals()]
if _missing:
    raise RuntimeError(f"Run the Phase 2 setup first. Missing: {_missing}")

FIX_DIR = f"{BASE}/finishing_audit"
os.makedirs(FIX_DIR, exist_ok=True)
_audit_started = time.time()


# ============================================================
# 1. Check identity decoding and repair pyramid construction.
# ============================================================
def staged_style_maps(Z, L, H, W, Cc):
    """Match the successive downsampling used by _pyr()."""
    sizes = sorted(set(FMS), reverse=True)
    if sizes[0] != L or H != L or W != L:
        raise ValueError("This helper expects the notebook's square finest grid.")

    current = Z.reshape(H, W, Cc).permute(2, 0, 1).unsqueeze(0)
    maps = {}
    for size in sizes:
        if current.shape[-2:] != (size, size):
            current = F.interpolate(
                current, size=(size, size),
                mode="bilinear", align_corners=False,
            )
        maps[size] = current
    return maps


def direct_style_maps(Z, L, H, W, Cc):
    """Original transported-field pyramid, retained for diagnosis."""
    finest = Z.reshape(H, W, Cc).permute(2, 0, 1).unsqueeze(0)
    return {
        size: (
            finest if size == L else
            F.interpolate(
                finest, size=(size, size),
                mode="bilinear", align_corners=False,
            )
        )
        for size in set(FMS)
    }


@torch.no_grad()
def identity_audit(pair_ids):
    rows = []

    for k in pair_ids:
        d = get_d(int(k))
        L, Cc, H, W, source, _ = _grids(d)

        native_maps = build_style_maps(d["bf_a"], d["si_a"], FMS)
        direct_maps = direct_style_maps(source, L, H, W, Cc)
        staged_maps = staged_style_maps(source, L, H, W, Cc)

        native = interp_lin_pixel(d, 0.0)
        direct = decode_with_style_maps(d["blob_a"], direct_maps)
        staged = decode_with_style_maps(d["blob_a"], staged_maps)

        def map_relative_error(candidate):
            errors = []
            for size in set(FMS):
                reference = native_maps[size].double()
                error = candidate[size].double() - reference
                errors.append(float(
                    error.norm() / reference.norm().clamp_min(1e-12)
                ))
            return max(errors)

        # Decoder output is in [-1, 1]; express MAE in 8-bit intensity units.
        rows.append({
            "k": int(k),
            "direct_map_rel_error": map_relative_error(direct_maps),
            "staged_map_rel_error": map_relative_error(staged_maps),
            "direct_MAE_255": float((direct - native).abs().mean() * 127.5),
            "staged_MAE_255": float((staged - native).abs().mean() * 127.5),
            "direct_LPIPS": lpips_d(to_pil(direct), to_pil(native)),
            "staged_LPIPS": lpips_d(to_pil(staged), to_pil(native)),
        })

    return pd.DataFrame(rows)


# Five reproducibly spaced pairs from the 200-pair ablation.
_audit_pool = [int(p["k"]) for p in MAN_ALL[:200]]
_audit_indices = np.unique(
    np.linspace(0, len(_audit_pool) - 1, min(5, len(_audit_pool))).astype(int)
)
_audit_ids = [_audit_pool[i] for i in _audit_indices]

print("=== Identity-field decoding check ===")
IDENTITY_AUDIT = identity_audit(_audit_ids)
print(IDENTITY_AUDIT.round(7).to_string(index=False))
IDENTITY_AUDIT.to_csv(f"{FIX_DIR}/identity_audit.csv", index=False)

# Numerical agreement checks, not perceptual-significance thresholds.
_staged_ok = bool(
    (IDENTITY_AUDIT.staged_map_rel_error.max() < 1e-5)
    and (IDENTITY_AUDIT.staged_MAE_255.max() < 0.01)
)
_direct_mismatch = bool(
    (IDENTITY_AUDIT.direct_map_rel_error.max() > 1e-5)
    or (IDENTITY_AUDIT.direct_MAE_255.max() > 0.01)
)

if _staged_ok:
    # _dec() resolves this global helper when called.
    if "_pre_audit_sm_from_field" not in globals():
        _pre_audit_sm_from_field = sm_from_field
    sm_from_field = staged_style_maps

    print("\nSuccess: staged pyramid matches the ordinary reconstruction.")
    print("Patched sm_from_field for FUTURE calls to _dec/decode_field.")
else:
    print("\nWARNING: staged construction still does not reproduce the native path.")
    print("No pyramid patch applied; inspect identity_audit.csv before rerendering.")

if _direct_mismatch:
    print(
        "\nOriginal direct-resize path differs from the native reconstruction.\n"
        "Existing images retain that difference. Corrected KID below still scores\n"
        "those EXISTING images; it does not remove this rendering confound.\n"
        "The equal-sample hard/soft comparison shares a rendering path, but\n"
        "linear-versus-transport and alignment interpretations need this resolved."
    )
else:
    print("\nNo identity discrepancy above the stated numerical tolerances on these pairs.")


# ============================================================
# 2. Correct the shared reference term in KID.
# ============================================================
REF = np.asarray(ref_inc, dtype=np.float64)
if REF.ndim != 2 or len(REF) < 3 or not np.isfinite(REF).all():
    raise ValueError("Invalid Inception reference features.")

_nref = len(REF)
_ref_kernel = (REF @ REF.T / REF.shape[1] + 1.0) ** 3

_old_kyy = (_ref_kernel.sum() - _nref) / (_nref * (_nref - 1))
KYY = (
    (_ref_kernel.sum() - np.trace(_ref_kernel))
    / (_nref * (_nref - 1))
)
_KID_OFFSET = float(_old_kyy - KYY)
del _ref_kernel

print("\n=== KID correction ===")
print("Reference feature shape:", REF.shape)
print(f"Old absolute KID minus corrected KID = {_KID_OFFSET:.10f}")
print("This shared offset cancels from within-experiment KID differences.")


def kid_components(features):
    """Full unbiased KID estimate and delete-one-generated-pair estimates."""
    features = np.asarray(features, dtype=np.float64)
    n, dim = features.shape

    if n < 3 or dim != REF.shape[1] or not np.isfinite(features).all():
        raise ValueError("Need >=3 finite feature vectors of matching dimension.")

    kxx = (features @ features.T / dim + 1.0) ** 3
    np.fill_diagonal(kxx, 0.0)
    total = kxx.sum()
    row_sums = kxx.sum(axis=1)

    cross = ((features @ REF.T / dim + 1.0) ** 3).mean(axis=1)

    full = total / (n * (n - 1)) + KYY - 2.0 * cross.mean()
    leave_one_out = (
        (total - 2.0 * row_sums) / ((n - 1) * (n - 2))
        + KYY
        - 2.0 * (cross.sum() - cross) / (n - 1)
    )
    return float(full), leave_one_out


def kid_ub(features):
    # Repairs the function used by the original notebook as well.
    return kid_components(features)[0]


def paired_jackknife(full_a, loo_a, full_b, loo_b):
    """Approximate normal 95% CI for paired KID difference A - B."""
    differences = np.asarray(loo_a) - np.asarray(loo_b)
    n = len(differences)
    estimate = float(full_a - full_b)
    se = np.sqrt(
        (n - 1) / n * np.sum((differences - differences.mean()) ** 2)
    )
    radius = norm.ppf(0.975) * se
    return estimate, float(estimate - radius), float(estimate + radius)


# ============================================================
# 3. Recover features from saved images, including resumed runs.
#    No encoder, transport solve or generator is used here.
# ============================================================
def saved_features(pair_ids, directory, batch_size=32):
    paths = [f"{directory}/{int(k):04d}.png" for k in pair_ids]
    missing = [p for p in paths if not os.path.isfile(p)]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} images missing; first: {missing[0]}"
        )

    chunks = []
    for start in range(0, len(paths), batch_size):
        images = []
        for path in paths[start:start + batch_size]:
            with Image.open(path) as im:
                images.append(at128(im.convert("RGB")))
        chunks.append(inc_feats(images, bs=batch_size))

    return np.concatenate(chunks).astype(np.float64)


def old_fid_lookup(path, kind):
    """FID is unaffected by the KID correction; retain existing measurements."""
    if not os.path.isfile(path):
        return {}

    if kind == "json":
        with open(path) as handle:
            records = json.load(handle)["table"]
        return {
            row["rule"]: float(row["FID"])
            for row in records if "FID" in row
        }

    frame = pd.read_csv(path, index_col=0)
    return frame["FID"].to_dict() if "FID" in frame else {}


def rebuild_kid_table(pair_ids, directories, old_fids, tag):
    pair_ids = sorted(set(int(k) for k in pair_ids))
    features, estimates, jackknives = {}, {}, {}

    for rule, directory in directories.items():
        print(f"{tag}: reading {len(pair_ids)} saved images for {rule}...")
        features[rule] = saved_features(pair_ids, directory)
        estimates[rule], jackknives[rule] = kid_components(features[rule])

    table = pd.DataFrame({
        "n": {rule: len(pair_ids) for rule in directories},
        "KID_corrected": estimates,
        "FID_existing": {
            rule: old_fids.get(rule, np.nan) for rule in directories
        },
    })
    table.index.name = "rule"

    # Retain recovered features and their exact row ordering for later scoring.
    np.savez_compressed(
        f"{FIX_DIR}/{tag}_inception_features.npz",
        pair_ids=np.asarray(pair_ids),
        **features,
    )
    return table, features, estimates, jackknives


def comparison_table(estimates, jackknives, comparisons):
    rows = []
    for a, b in comparisons:
        delta, lo, hi = paired_jackknife(
            estimates[a], jackknives[a], estimates[b], jackknives[b]
        )
        rows.append({
            "comparison": f"{a} - {b}",
            "delta_KID": delta,
            "JK95_low": lo,
            "JK95_high": hi,
            "reading": (
                "favors first" if hi < 0
                else "favors second" if lo > 0
                else "interval includes zero"
            ),
        })
    return pd.DataFrame(rows)


print(
    "\nIntervals below are approximate paired delete-one jackknife intervals,\n"
    "conditional on the fixed reference and treating pairs as sampling units.\n"
    "They are not the previous m-of-n subsampling bands and are not adjusted\n"
    "for testing multiple epsilon settings or dependence from shared sources."
)

# --- TM ---
_tm_stats = pd.read_csv(f"{BASE}/tm_coupling_stats.csv")
_tm_ot_rules = sorted(
    [r for r in _tm_stats.rule.unique() if r.startswith("ot_r")],
    key=lambda r: -float(r[4:]),
)
_tm_rules = ["lerp", "mean"] + _tm_ot_rules + ["hard"]
_tm_dirs = {r: f"{SCALE}/tmc/{r}" for r in _tm_rules}
if os.path.isdir(f"{SCALE}/mid"):
    _tm_dirs["mid_pipeline"] = f"{SCALE}/mid"

_tm_fids = old_fid_lookup(f"{BASE}/tm_coupling_summary.json", "json")
TM_FIXED, TM_FEATURES_FIXED, _tm_est, _tm_loo = rebuild_kid_table(
    KS120, _tm_dirs, _tm_fids, "tm"
)

_tm_comparisons = [
    (r, "lerp") for r in _tm_dirs if r != "lerp"
] + [(r, "mean") for r in _tm_ot_rules + ["hard"]]

TM_COMPARISONS_FIXED = comparison_table(
    _tm_est, _tm_loo, _tm_comparisons
)

print("\n=== Corrected TM absolute scores ===")
print(TM_FIXED.round(6).to_string())
print("\n=== TM paired KID differences: negative favors FIRST method ===")
print(TM_COMPARISONS_FIXED.round(6).to_string(index=False))

TM_FIXED.to_csv(f"{FIX_DIR}/tm_scores_corrected.csv")
TM_COMPARISONS_FIXED.to_csv(
    f"{FIX_DIR}/tm_kid_jackknife.csv", index=False
)

# Report convergence maxima, rather than just median residuals.
_tm_convergence = _tm_stats[
    _tm_stats.rule.str.startswith("ot_")
].groupby("rule").agg(
    max_marginal_error=("marg_err", "max"),
    pairs_above_tolerance=("marg_err", lambda s: int((s > 1e-3).sum())),
    nonfinite_errors=("marg_err", lambda s: int((~np.isfinite(s)).sum())),
)
print("\n=== TM convergence ===")
print(_tm_convergence.to_string())


# --- Equal-sample CNT ---
_eq_ids = [int(p["k"]) for p in MAN_ALL[:200]]
_eq_rules = ["hard", "ot_e0.005", "ot_e0.05"]
_eq_dirs = {
    r: f"{OUT}/eqsample/{r}_t0.5" for r in _eq_rules
}
_eq_fids = old_fid_lookup(f"{BASE}/eqsample_summary.csv", "csv")

CNT_FIXED, CNT_FEATURES_FIXED, _eq_est, _eq_loo = rebuild_kid_table(
    _eq_ids, _eq_dirs, _eq_fids, "cnt_equal_sample"
)
CNT_COMPARISONS_FIXED = comparison_table(
    _eq_est, _eq_loo,
    [
        ("hard", "ot_e0.05"),
        ("hard", "ot_e0.005"),
        ("ot_e0.005", "ot_e0.05"),
    ],
)

print("\n=== Corrected equal-sample CNT absolute scores ===")
print(CNT_FIXED.round(6).to_string())
print("\n=== CNT paired KID differences: negative favors FIRST method ===")
print(CNT_COMPARISONS_FIXED.round(6).to_string(index=False))

CNT_FIXED.to_csv(f"{FIX_DIR}/cnt_scores_corrected.csv")
CNT_COMPARISONS_FIXED.to_csv(
    f"{FIX_DIR}/cnt_kid_jackknife.csv", index=False
)

_eq_raw = pd.read_csv(f"{BASE}/eqsample_scores.csv")
_eq_convergence = _eq_raw[
    _eq_raw.rule.str.startswith("ot_")
].drop_duplicates(["k", "rule"]).groupby("rule").agg(
    max_marginal_error=("marg_err", "max"),
    pairs_above_tolerance=("marg_err", lambda s: int((s > 1e-3).sum())),
    nonfinite_errors=("marg_err", lambda s: int((~np.isfinite(s)).sum())),
)
print("\n=== CNT convergence ===")
print(_eq_convergence.to_string())

# Check endpoint/balance evidence without changing its anchor convention.
print(
    "\nPlacement and target-gap scores remain as previously computed:\n"
    "their anchors are CNT endpoint reconstructions, including for TM.\n"
    "This audit corrects KID; it does not change that evaluation convention."
)

_report = {
    "reference_count": int(len(REF)),
    "old_KID_minus_corrected_KID": _KID_OFFSET,
    "identity_test_pairs": [int(k) for k in _audit_ids],
    "original_pyramid_mismatch": _direct_mismatch,
    "staged_pyramid_matches_native": _staged_ok,
    "future_field_decoding_patched": _staged_ok,
    "existing_images_rerendered": False,
    "uncertainty": (
        "Approximate paired delete-one jackknife normal 95% intervals; "
        "fixed reference; pairs treated as sampling units; no multiplicity adjustment"
    ),
}
with open(f"{FIX_DIR}/audit_report.json", "w") as handle:
    json.dump(_report, handle, indent=2)

print(f"\nCompleted in {time.time() - _audit_started:.0f}s.")
print("Corrected outputs:", FIX_DIR)

if _direct_mismatch:
    print(
        "\nACTION NEEDED BEFORE FREEZING THE PAPER:\n"
        "Inspect the identity-error magnitudes above. Standardize the decoder\n"
        "pyramid and rerender affected field-versus-linear/alignment comparisons.\n"
        "Use NEW output directories and score CSVs: the original resume logic\n"
        "would otherwise silently reuse images generated with the old pyramid."
    )
elif not _staged_ok:
    print("\nACTION NEEDED: investigate the identity check before changing rendering.")
else:
    print(
        "\nIdentity check passed on the tested pairs. Use the corrected KID\n"
        "summaries and the explicitly labeled jackknife intervals."
    )

In [ ]:
# ============================================================
# FINAL CONSISTENT-PYRAMID RERUN
# Run after the audit cell. No restart.
#
# CNT: same 200 pairs, consistent decoder pyramid:
#   linear / hard1024 / OT1024 eps=.005 / OT1024 eps=.05 /
#   OT4096 eps=.05
#   Midpoints and transported endpoints.
#
# Alignment: same 40 images, shifts and coefficients as before.
#
# New caches only. Resume by rerunning THIS cell.
# ============================================================
import os
import json
import time
import hashlib
import numpy as np
import pandas as pd
import torch
from PIL import Image

_required = [
    "BASE", "OUT", "LOCAL", "MAN_ALL", "FMS", "REF",
    "staged_style_maps", "get_d", "_grids", "_sub", "_dec",
    "sink_log_t", "linear_sum_assignment", "score_image",
    "kid_components", "paired_jackknife", "to_pil",
    "seamless", "shift", "hf_energy", "cnt_load",
    "get_gen_inputs", "decode_with_style_maps", "build_style_maps",
    "field", "zt", "dino_feats", "at128", "realism",
    "MANI_DINO", "lpips_d",
]
_missing = [v for v in _required if v not in globals()]
if _missing:
    raise RuntimeError(f"Missing setup/audit definitions: {_missing}")

if not globals().get("_staged_ok", False):
    raise RuntimeError("The staged identity check must pass before this rerun.")

# Intentional patch: all field decodes now use successive downsampling.
sm_from_field = staged_style_maps

FINAL_DIR = f"{BASE}/consistent_pyramid_v1"
os.makedirs(FINAL_DIR, exist_ok=True)


def run_consistent_rerun():
    started = time.time()
    pair_ids = [int(p["k"]) for p in MAN_ALL[:200]]
    rules = [
        "linear", "hard1024",
        "ot1024_e0.005", "ot1024_e0.05", "ot4096_e0.05",
    ]
    times = [0.5, 1.0]

    config = {
        "version": 1,
        "pyramid": "successive bilinear downsampling, align_corners=False",
        "pairs": pair_ids,
        "rules": rules,
        "times": times,
        "transport": "float64 log-domain, relative marginal tolerance 1e-3",
        "sampling": "existing _sub with pair ID / alignment image index",
        "reference_shape": list(REF.shape),
        "reference_sha256": hashlib.sha256(
            np.ascontiguousarray(REF).tobytes()
        ).hexdigest(),
    }
    config_path = f"{FINAL_DIR}/config.json"
    if os.path.isfile(config_path):
        with open(config_path) as f:
            if json.load(f) != config:
                raise RuntimeError(
                    "Cache configuration differs. Use a new FINAL_DIR."
                )
    else:
        with open(config_path, "w") as f:
            json.dump(config, f, indent=2)

    def paths(rule, t, k):
        directory = f"{FINAL_DIR}/cnt/{rule}_t{t}"
        os.makedirs(directory, exist_ok=True)
        stem = f"{directory}/{k:04d}"
        return stem + ".png", stem + ".npz"

    def complete(rule, t, k):
        png, npz = paths(rule, t, k)
        return os.path.isfile(png) and os.path.isfile(npz)

    # --------------------------------------------------------
    # A. Rerender CNT with identical pyramid construction.
    # --------------------------------------------------------
    print("=== CNT: consistent-pyramid generation ===")

    for index, k in enumerate(pair_ids):
        if all(complete(r, t, k) for r in rules for t in times):
            continue

        d = get_d(k)
        L, channels, height, width, source, target = _grids(d)
        n = len(source)

        source_ids = _sub(n, k, "s")
        target_ids = _sub(n, k, "t")
        if min(len(source_ids), len(target_ids)) < 4096:
            raise ValueError("Expected 4096 samples from _sub().")

        # Both sample counts share the same sample ordering.
        for count in [1024, 4096]:
            src = source[source_ids[:count]]
            tgt = target[target_ids[:count]]
            nearest = torch.cdist(source, src).argmin(1)
            cost = torch.cdist(src.double(), tgt.double()).square()

            if count == 1024:
                row, col = linear_sum_assignment(cost.cpu().numpy())
                hard = torch.empty_like(src)
                hard[torch.as_tensor(row, device=src.device)] = (
                    tgt[torch.as_tensor(col, device=tgt.device)]
                )
                targets = {
                    "linear": (
                        target,
                        {"samples": float(n), "marg_err": np.nan},
                    ),
                    "hard1024": (
                        hard[nearest],
                        {"samples": 1024.0, "marg_err": 0.0},
                    ),
                }
                epsilons = [0.005, 0.05]
            else:
                targets = {}
                epsilons = [0.05]

            for epsilon in epsilons:
                name = f"ot{count}_e{epsilon}"
                if all(complete(name, t, k) for t in times):
                    continue

                plan, error, iterations = sink_log_t(
                    cost, epsilon, tol=1e-3, max_it=5000
                )
                if not np.isfinite(error) or error > 1e-3:
                    raise RuntimeError(
                        f"Nonconverged plan: pair={k}, rule={name}, "
                        f"relative error={error:.6g}. "
                        "Increase max_it and resume."
                    )

                conditional = plan / plan.sum(1, keepdim=True)
                mapped = (conditional @ tgt.double()).float()
                targets[name] = (
                    mapped[nearest],
                    {
                        "samples": float(count),
                        "marg_err": float(error),
                        "iterations": float(iterations),
                        "var_ret": float(
                            mapped.double().var(0).sum()
                            / tgt.double().var(0).sum().clamp_min(1e-30)
                        ),
                    },
                )

            for rule, (mapped_full, diagnostics) in targets.items():
                for t in times:
                    if complete(rule, t, k):
                        continue

                    png, npz = paths(rule, t, k)
                    with torch.no_grad():
                        output = _dec(
                            d, (1 - t) * source + t * mapped_full,
                            L, height, width, channels,
                        )
                    pil = to_pil(output)
                    pil.save(png)
                    scores, inception, _ = score_image(k, pil)

                    # Save features together with scores, making resume safe.
                    np.savez_compressed(
                        npz, inc=inception,
                        **scores, **diagnostics,
                    )

            del cost, nearest, targets

        if (index + 1) % 20 == 0 or index + 1 == len(pair_ids):
            print(
                f"CNT {index + 1}/{len(pair_ids)} | "
                f"{time.time() - started:.0f}s"
            )

    # --------------------------------------------------------
    # B. Read cached features and compute corrected summaries.
    # --------------------------------------------------------
    rows, features = [], {}
    for rule in rules:
        midpoint_features = []
        for k in pair_ids:
            for t in times:
                _, npz = paths(rule, t, k)
                with np.load(npz) as saved:
                    record = {
                        key: float(saved[key])
                        for key in saved.files if key != "inc"
                    }
                    if t == 0.5:
                        midpoint_features.append(saved["inc"].copy())

                rows.append({"k": k, "rule": rule, "t": t, **record})

        features[rule] = np.stack(midpoint_features).astype(np.float64)

    scores = pd.DataFrame(rows)
    scores["balance"] = (scores["place"] - 0.5).abs()
    scores.to_csv(f"{FINAL_DIR}/cnt_per_pair.csv", index=False)

    # Exact low-rank covariance cross-term; avoids a 2048x2048 sqrtm.
    reference_mean = REF.mean(0)
    reference_centered = (
        (REF - reference_mean) / np.sqrt(len(REF) - 1)
    )
    reference_trace = np.sum(reference_centered ** 2)

    def feature_fid(x):
        mean = x.mean(0)
        centered = (x - mean) / np.sqrt(len(x) - 1)
        cross = centered @ reference_centered.T
        cross_trace = np.linalg.svd(cross, compute_uv=False).sum()
        value = (
            np.sum((mean - reference_mean) ** 2)
            + np.sum(centered ** 2)
            + reference_trace - 2 * cross_trace
        )
        return float(max(value, 0.0))

    summary = (
        scores[scores.t == 0.5]
        .groupby("rule")[["real_inc", "real_dino", "place", "balance"]]
        .median().reindex(rules)
    )
    estimates, leave_one_out = {}, {}

    for rule in rules:
        estimates[rule], leave_one_out[rule] = kid_components(features[rule])
        summary.loc[rule, "KID"] = estimates[rule]
        summary.loc[rule, "FID"] = feature_fid(features[rule])

    endpoint = (
        scores[scores.t == 1.0]
        .groupby("rule")[["dB", "real_dino"]].median()
        .rename(columns={
            "dB": "target_gap_t1",
            "real_dino": "real_dino_t1",
        })
    )
    summary = summary.join(endpoint)
    summary.to_csv(f"{FINAL_DIR}/cnt_summary.csv")

    comparisons = [
        ("ot4096_e0.05", "linear"),
        ("ot1024_e0.05", "linear"),
        ("ot1024_e0.005", "linear"),
        ("hard1024", "ot1024_e0.05"),
        ("hard1024", "ot1024_e0.005"),
        ("ot1024_e0.005", "ot1024_e0.05"),
    ]
    difference_rows = []
    for a, b in comparisons:
        delta, low, high = paired_jackknife(
            estimates[a], leave_one_out[a],
            estimates[b], leave_one_out[b],
        )
        difference_rows.append({
            "comparison": f"{a} - {b}",
            "delta_KID": delta,
            "JK95_low": low,
            "JK95_high": high,
        })

    differences = pd.DataFrame(difference_rows)
    differences.to_csv(f"{FINAL_DIR}/cnt_kid_intervals.csv", index=False)

    print("\n=== CNT final scores: identical pyramid construction ===")
    print(summary.round(6).to_string())
    print("\n=== Paired KID differences: negative favors FIRST method ===")
    print(differences.round(6).to_string(index=False))
    print(
        "Intervals: approximate paired jackknife; fixed reference; "
        "pairs treated as units; no multiple-comparison adjustment."
    )

    # --------------------------------------------------------
    # C. Rerun alignment with the corrected common pyramid.
    # --------------------------------------------------------
    import glob

    files = sorted(glob.glob(f"{LOCAL}/*/*.jpg"))
    order = np.random.RandomState(7).permutation(len(files))
    selected = []
    for position in order:
        path = files[position]
        with Image.open(path) as im:
            eligible = min(im.size) >= 256
        if eligible:
            selected.append(path)
        if len(selected) == 40:
            break
    if len(selected) != 40:
        raise RuntimeError("Could not recover 40 eligible alignment images.")

    manifest_path = f"{FINAL_DIR}/alignment_sources.json"
    if os.path.isfile(manifest_path):
        with open(manifest_path) as f:
            if json.load(f) != selected:
                raise RuntimeError(
                    "Alignment source selection differs from this cache. "
                    "Use a new FINAL_DIR."
                )
    else:
        with open(manifest_path, "w") as f:
            json.dump(selected, f, indent=2)

    alignment_dir = f"{FINAL_DIR}/alignment"
    os.makedirs(alignment_dir, exist_ok=True)
    shifts = [0, 4, 16, 64]
    alignment_times = [0.25, 0.5, 0.75]
    alignment_rules = ["linear", "gauss", "ot"]

    def alignment_path(i, s, rule, t):
        return f"{alignment_dir}/img{i:02d}_s{s}_{rule}_t{t}"

    print("\n=== Alignment: consistent-pyramid generation ===")
    for i, source_path in enumerate(selected):
        expected = [
            alignment_path(i, s, r, t) + ".json"
            for s in shifts for r in alignment_rules for t in alignment_times
        ]
        if all(os.path.isfile(p) for p in expected):
            continue

        A = seamless(source_path)

        # Encode A once and retain the exact same source conditioning.
        ba, bfa, sia = get_gen_inputs(cnt_load(A))
        with torch.no_grad():
            reconstruction = to_pil(decode_with_style_maps(
                ba, build_style_maps(bfa, sia, FMS)
            ))
        reference_dino = dino_feats([at128(reconstruction)])[0]
        reference_hf = hf_energy(reconstruction)

        if i < 3:
            reconstruction.save(f"{alignment_dir}/img{i:02d}_reconA.png")

        for s in shifts:
            B = shift(A, s)
            bb, bfb, sib = get_gen_inputs(cnt_load(B))
            d = {
                "blob_a": ba, "bf_a": bfa, "si_a": sia,
                "blob_b": bb, "bf_b": bfb, "si_b": sib,
            }
            L, channels, height, width, source, target = _grids(d)

            # Preserve the original Gaussian construction and sampling.
            gaussian_field = field(d, "gauss", i)

            # Same OT objective; use log-domain solve and check convergence.
            ot_field = field(d, "ot", i, plan_kind="log")
            residual = float(ot_field["info"]["res"])
            if not np.isfinite(residual) or residual > 1e-3:
                raise RuntimeError(
                    f"Alignment OT not converged: image={i}, shift={s}, "
                    f"relative error={residual:.6g}"
                )

            targets = {
                "linear": target,
                "gauss": gaussian_field["U"],
                "ot": ot_field["U"],
            }

            if i < 3:
                B.save(f"{alignment_dir}/img{i:02d}_s{s}_B.png")

            for rule, mapped_full in targets.items():
                for t in alignment_times:
                    stem = alignment_path(i, s, rule, t)
                    if os.path.isfile(stem + ".json"):
                        continue

                    with torch.no_grad():
                        output = _dec(
                            d, (1 - t) * source + t * mapped_full,
                            L, height, width, channels,
                        )
                    pil = to_pil(output)
                    fd = dino_feats([at128(pil)])[0]
                    record = {
                        "img": i, "shift": s, "rule": rule, "t": t,
                        "dino_dev": float(np.linalg.norm(fd - reference_dino)),
                        "lpips_recA": lpips_d(pil, reconstruction),
                        # Fraction-of-spectrum ratio, not absolute HF energy.
                        "hf_fraction_ratio": float(
                            hf_energy(pil) / (reference_hf + 1e-9)
                        ),
                        "real_dino": float(realism(fd[None], MANI_DINO)[0]),
                        "marg_err": residual if rule == "ot" else None,
                    }
                    if i < 3:
                        pil.save(stem + ".png")
                    with open(stem + ".json", "w") as f:
                        json.dump(record, f, indent=2)

        if (i + 1) % 5 == 0 or i + 1 == len(selected):
            print(f"Alignment {i + 1}/{len(selected)}")

    alignment_rows = []
    for i in range(len(selected)):
        for s in shifts:
            for rule in alignment_rules:
                for t in alignment_times:
                    with open(alignment_path(i, s, rule, t) + ".json") as f:
                        alignment_rows.append(json.load(f))

    alignment = pd.DataFrame(alignment_rows)
    alignment.to_csv(f"{FINAL_DIR}/alignment_per_image.csv", index=False)

    print("\n=== Alignment at t=0.5: medians ===")
    for metric in ["dino_dev", "lpips_recA", "hf_fraction_ratio", "real_dino"]:
        print(f"\n{metric}")
        print(
            alignment[alignment.t == 0.5].pivot_table(
                index="shift", columns="rule",
                values=metric, aggfunc="median",
            ).round(5).to_string()
        )

    print(
        "\nInterpretation: lower DINO deviation/LPIPS means closer to the\n"
        "source reconstruction under that observer. HF fraction ratio near\n"
        "1 means preservation of that spectral fraction; larger is not\n"
        "automatically better. These observers are not exact texture invariants."
    )
    print(f"\nFINISHED in {time.time() - started:.0f}s.")
    print("Saved:", FINAL_DIR)
    print("TM images and results were not changed.")

    return summary, differences, alignment


FINAL_CNT_SUMMARY, FINAL_CNT_DIFFERENCES, FINAL_ALIGNMENT = (
    run_consistent_rerun()
)